####Requirement:
We have colleced fire calls data file sf-fire-calls.csv.
1. Read the data file
2. Load it into a table for analysis
3. Verify all 175296 records are loaded correctly
4. The table is predefined as below

In [0]:
fire_calls_df = (spark.read.format('csv')
                 .option("header","true")
    .option("inferSchema" , "true")
 .load('/Volumes/dev/spark_db/datasets/spark_programming/data/sf-fire-calls.csv')
)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS dev.spark_db.sf_fire_calls (
  CallNumber INT, UnitID STRING, IncidentNumber INT, CallType STRING, CallDate DATE,
  WatchDate DATE, CallFinalDisposition STRING, AvailableDtTm TIMESTAMP, Address STRING,
  City STRING, Zipcode STRING, Battalion STRING, StationArea STRING, Box STRING,
  OriginalPriority STRING, Priority STRING, FinalPriority STRING, ALSUnit BOOLEAN,
  CallTypeGroup STRING, NumAlarms INT, UnitType STRING, UnitSequenceInCallDispatch INT,
  FirePreventionDistrict STRING, SupervisorDistrict STRING, Neighborhood STRING,
  Location STRING, RowID STRING, Delay DOUBLE);

In [0]:
display(fire_calls_df)

In [0]:
#row count 
fire_calls_df.count()

In [0]:
#schema
fire_calls_df.printSchema()

In [0]:
#schema
from pyspark.sql.functions import to_timestamp,expr

fire_calls_df_updated =fire_calls_df.withColumns(

    {
      "AvailableDtTm":  to_timestamp("AvailableDtTm", "MM/dd/yyyy hh:mm:ss a"),
      "FinalPriority": expr("CAST(FinalPriority AS STRING)"),
        "Zipcode": expr("CAST(Zipcode AS STRING)")
    }
)

In [0]:
fire_calls_df_updated.printSchema()

In [0]:
fire_calls_df_updated.display()

In [0]:
# load into table

fire_calls_df_updated.write.mode('overwrite').saveAsTable('dev.spark_db.sf_fire_calls')

In [0]:
%sql
SELECT * FROM dev.spark_db.sf_fire_calls

In [0]:
#top 3 zipcodes with the most calls

In [0]:
%sql
SELECT CallType, Zipcode, COUNT(*) as total_calls
FROM dev.spark_db.sf_fire_calls
GROUP BY CallType, Zipcode
ORDER BY total_calls DESC
LIMIT 3

In [0]:
#sql equivalent 

#step 1 : read table
df =spark.read.table("dev.spark_db.sf_fire_calls")

In [0]:
#transformations
from pyspark.sql import functions as F 

df = df.filter("CallType is NOT NULL")\
            .groupBy('CallType',"Zipcode")\
            .count()\
            .orderBy(F.col("count")\
            .desc())

###### Dataframe concepts

1. Spark creates optimised query plan(parsed query plan->analysed logical plan-->iptimised logilca plan-->phyiscal plan-->cost based mpdel optimisation-->RDDs(convert code to scala)
2.




1) to see the execution plan --> you can do EXPLAIN plan

In [0]:
df.explain(mode='extended')

In [0]:
df = df.filter("CallType is NOT NULL")\
            .groupBy('CallType',"Zipcode")\
            .count()\
            .orderBy(F.col("count")\
            .desc())

In [0]:
%sql
SELECT * FROM dev.spark_db.sf_fire_calls

In [0]:
%sql
--how many distinct types of calls were made to the fire department
SELECT COUNT(DISTINCT CallType) AS number_of_call_types
FROM dev.spark_db.sf_fire_calls

In [0]:
%sql
-- distinct types of calls were made to the fire department
SELECT DISTINCT CallType AS call_types
FROM dev.spark_db.sf_fire_calls
WHERE CallType IS NOT NULL

In [0]:
# responses for delayed time >5min

fire_calls_df_updated = spark.read.table('dev.spark_db.sf_fire_calls')
from pyspark.sql.functions import col
delayed_response_records = fire_calls_df_updated.filter(col('Delay') >5)

delayed_response_records.display()

In [0]:
# What were the most common call types

most_common_call_types = (fire_calls_df_updated\
    .groupBy('CallType')\
        .count()\
            .orderBy(col('count').desc()) )


In [0]:
most_common_call_types.display()

In [0]:
fire_calls_df_updated.select(col('CallType')).distinct().count()

In [0]:
# What San Francisco neighbourhoods are in the zip codes 94102 and 94103
neighbourhoods_in_zip = fire_calls_df_updated.select(col('Neighborhood'), col('Zipcode'))\
            .filter(col('Zipcode').isin(['94102' ,'94103']))


neighbourhoods_in_zip.display()